<a href="https://colab.research.google.com/github/Hala-sala7/telehealth-data-pipeline/blob/main/analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Telehealth Data Pipeline — Analysis
**Author:** Hala Salah

This notebook uses the **gold layer** built by the pipeline to answer business
questions for a virtual health clinic: appointment trends, no-shows, revenue,
and lab results.

## 1. Setup (Colab only)
Clone the repository, install the requirements and run the pipeline.
Skip this cell if you are running the notebook locally after `python run_pipeline.py`.

In [ ]:
!git clone https://github.com/Hala-sala7/telehealth-data-pipeline.git
%cd telehealth-data-pipeline
!pip install -r requirements.txt -q
!python run_pipeline.py

## 2. Imports

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

## 3. Connect to the Warehouse
Open the DuckDB file in read-only mode and list the tables in each layer.

In [ ]:
con = duckdb.connect('data/warehouse.duckdb', read_only=True)

con.sql("""
    SELECT table_schema, table_name
    FROM information_schema.tables
    WHERE table_schema IN ('bronze', 'silver', 'gold')
    ORDER BY table_schema, table_name
""").df()

## 4. How Much Did Cleaning Remove?
Compare row counts between bronze (raw) and silver (clean).

In [ ]:
for table in ['patients', 'appointments', 'lab_results']:
    bronze = con.sql(f"SELECT count(*) FROM bronze.{table}").fetchone()[0]
    silver = con.sql(f"SELECT count(*) FROM silver.stg_{table}").fetchone()[0]
    removed = bronze - silver
    print(f"{table}: bronze = {bronze}, silver = {silver}, removed = {removed} ({round(removed / bronze * 100, 1)}%)")

## 5. Monthly Appointments and Revenue

In [ ]:
monthly = con.sql("SELECT * FROM gold.agg_monthly_appointments").df()
monthly['month'] = pd.to_datetime(monthly['month'])
monthly = monthly.set_index('month')
monthly.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

monthly['total_appointments'].plot(ax=axes[0], title='Appointments per Month')
axes[0].set_ylabel('Number of Appointments')

monthly['revenue_usd'].astype(float).plot(ax=axes[1], title='Revenue per Month (USD)', color='seagreen')
axes[1].set_ylabel('Revenue (USD)')

plt.tight_layout()
plt.show()

print("Total revenue:", round(monthly['revenue_usd'].astype(float).sum()), "USD")
print("Best month:", monthly['revenue_usd'].astype(float).idxmax().strftime('%B %Y'))

## 6. No-Show Rate
Which providers and appointment types have the most no-shows?

In [ ]:
no_shows = con.sql("""
    SELECT
        provider_name,
        count(*) AS appointments,
        round(100.0 * count(*) FILTER (WHERE status = 'no_show') / count(*), 1) AS no_show_rate_pct
    FROM silver.stg_appointments
    GROUP BY provider_name
    ORDER BY no_show_rate_pct DESC
""").df()

no_shows.set_index('provider_name')['no_show_rate_pct'].sort_values().plot(
    kind='barh', title='No-Show Rate by Provider (%)', color='indianred')
plt.xlabel('No-Show Rate (%)')
plt.show()

print("Overall no-show rate:", round(monthly['no_shows'].sum() / monthly['total_appointments'].sum() * 100, 1), "%")
no_shows

## 7. Patients by Age Group

In [ ]:
patients = con.sql("SELECT * FROM gold.dim_patients").df()

by_age = patients.groupby('age_group').agg(
    patients=('patient_id', 'count'),
    avg_appointments=('total_appointments', 'mean'),
    avg_revenue=('lifetime_revenue_usd', 'mean')
).round(1)

by_age['avg_revenue'].plot(kind='bar', title='Average Lifetime Revenue per Patient by Age Group')
plt.ylabel('Revenue (USD)')
plt.xticks(rotation=0)
plt.show()

by_age

## 8. Abnormal Lab Results
Share of results outside the normal range for each test.

In [ ]:
labs = con.sql("""
    SELECT
        test_name,
        count(*) AS results,
        round(100.0 * count(*) FILTER (WHERE result_flag <> 'normal') / count(*), 1) AS abnormal_rate_pct
    FROM silver.stg_lab_results
    GROUP BY test_name
    ORDER BY abnormal_rate_pct DESC
""").df()

labs.set_index('test_name')['abnormal_rate_pct'].sort_values().plot(
    kind='barh', title='Abnormal Lab Results by Test (%)', color='steelblue')
plt.xlabel('Abnormal Results (%)')
plt.show()

labs

In [ ]:
con.close()